In [ ]:
#Note:
#This notebook contains the initial local RAG implementation using ChromaDB.
#The final deployed solution uses Qdrant Cloud as shown in 05_qdrant_migration.ipynb and rag_pipeline.py.

In [3]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 478.89it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [5]:
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model
)

print(vectorstore._collection.count())

50000


In [6]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("Retriever ready")

Retriever ready


In [7]:
load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [8]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a Python Programming Assistant.

Use ONLY the provided context.

If the context contains information that answers the question,
provide a detailed answer.

If the context does NOT contain enough information,
respond exactly with:

"I could not find the answer in the provided knowledge base."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [9]:
THRESHOLD = 1.0

def ask_rag(query):

    docs_with_scores = vectorstore.similarity_search_with_score(
        query,
        k=5
    )

    # Safety check
    if not docs_with_scores:
        return {
            "answer": "I could not find the answer in the provided knowledge base.",
            "sources": []
        }

    best_score = docs_with_scores[0][1]

    # Threshold check
    if best_score > THRESHOLD:
        return {
            "answer": "I could not find the answer in the provided knowledge base.",
            "sources": []
        }

    docs = [doc for doc, score in docs_with_scores]

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    final_prompt = prompt.format(
        context=context,
        question=query
    )

    response = llm.invoke(final_prompt)

    return {
        "answer": response.content,
        "sources": [
            doc.metadata
            for doc in docs
        ]
    }

In [10]:
result = ask_rag(
    "What is a metaclass in Python?"
)

print(result["answer"])

for source in result["sources"]:
    print(source)

Metaclasses are classes whose instances are classes. In other words, a metaclass is a class that creates classes. They are used to customize the creation of classes, and can be used to implement complex class creation logic.

In Python, metaclasses are classes that inherit from the `type` class, which is the built-in metaclass that creates all classes. When you define a class, Python uses the `type` metaclass to create the class. However, you can define your own metaclass by inheriting from `type` and overriding its `__new__` method.

Metaclasses are useful for a variety of tasks, such as:

* Customizing the creation of classes
* Implementing complex class creation logic
* Creating classes dynamically
* Modifying the behavior of classes at creation time

Here is an example of a simple metaclass that prints a message when a class is created:
```python
class MyMetaclass(type):
    def __new__(meta, name, bases, namespace):
        print(f"Creating class {name}")
        return type.__new

In [11]:
result = ask_rag(
    "Who won FIFA World Cup 2022?"
)

print(result["answer"])

I could not find the answer in the provided knowledge base.


In [12]:
test_questions = [
    "What is a metaclass in Python?",
    "How do Python generators work?",
    "How do I execute shell commands from Python?",
    "What is Django?",
    "Who won FIFA World Cup 2022?",
    "What is the capital of France?"
]

for q in test_questions:

    docs = vectorstore.similarity_search_with_score(
        q,
        k=1
    )

    print("\n" + "="*60)
    print("Question:", q)

    if docs:
        print("Best Score:", docs[0][1])


Question: What is a metaclass in Python?
Best Score: 0.3853244185447693

Question: How do Python generators work?
Best Score: 0.5592827796936035

Question: How do I execute shell commands from Python?
Best Score: 0.5008023977279663

Question: What is Django?
Best Score: 0.637497067451477

Question: Who won FIFA World Cup 2022?
Best Score: 1.6260972023010254

Question: What is the capital of France?
Best Score: 1.4135003089904785


In [13]:
more_questions = [
    "What is a Python decorator?",
    "How do I sort a dictionary in Python?",
    "What is a lambda function?",
    "What is __name__ == '__main__'?",
    "How do I install pip?",
    "How do I merge dictionaries?",
    "What is recursion?",
    "How do I reverse a list in Python?",
    "Best restaurants in Mumbai",
    "Latest IPL winner",
]

for q in more_questions:

    docs = vectorstore.similarity_search_with_score(
        q,
        k=1
    )

    print("\n" + "=" * 60)
    print("Question:", q)

    if docs:
        print("Best Score:", docs[0][1])

        print("Source Metadata:")
        print(docs[0][0].metadata)

    else:
        print("No documents found")


Question: What is a Python decorator?
Best Score: 0.4444185197353363
Source Metadata:
{'question_id': 8328824, 'tags': 'python, design-patterns, decorator'}

Question: How do I sort a dictionary in Python?
Best Score: 0.41637933254241943
Source Metadata:
{'tags': 'python, sorting, dictionary', 'question_id': 613183}

Question: What is a lambda function?
Best Score: 0.49313977360725403
Source Metadata:
{'question_id': 5233508, 'tags': 'python, lambda'}

Question: What is __name__ == '__main__'?
Best Score: 0.8232536911964417
Source Metadata:
{'question_id': 4777031, 'tags': 'python'}

Question: How do I install pip?
Best Score: 0.5251675844192505
Source Metadata:
{'tags': 'python, python-3.x, pip', 'question_id': 24285508}

Question: How do I merge dictionaries?
Best Score: 0.6073599457740784
Source Metadata:
{'tags': 'python, dictionary', 'question_id': 10461531}

Question: What is recursion?
Best Score: 0.8419380187988281
Source Metadata:
{'question_id': 15128424, 'tags': 'python'}



In [14]:
result = ask_rag(
    "What is a Python decorator?"
)

print(result)

KeyboardInterrupt: 

In [ ]:
result = ask_rag(
    "Who won FIFA World Cup 2022?"
)

print(result)

{'answer': 'I could not find the answer in the provided knowledge base.', 'sources': []}


In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model
)

print("Vector Count:", vectorstore._collection.count())

c:\Users\bilal\Desktop\Crosstab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 413.82it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Count: 50000


In [ ]:
import os

def get_folder_size(folder):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size

size_mb = get_folder_size("chroma_db") / (1024 * 1024)

print(f"ChromaDB Size: {size_mb:.2f} MB")

ChromaDB Size: 781.84 MB
